# 16.3 隐私保护 (Privacy Protection)

保护训练数据和用户交互中的隐私信息，是AI系统合规和可信的基础。

本节涵盖：
- 差分隐私 (DP-SGD)
- 数据脱敏与匿名化
- 遗忘学习 (Machine Unlearning)
- 联邦学习与隐私计算
- LLM隐私风险评估
- 隐私保护最佳实践

## 1. 差分隐私 (Differential Privacy)

**差分隐私**提供数学上可证明的隐私保证：

$$\Pr[\mathcal{M}(D) \in S] \leq e^\epsilon \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$

含义：数据集D和D'（相差一条记录）的输出分布几乎不可区分。

### DP-SGD核心步骤
1. **梯度裁剪**：限制每个样本的梯度范数，约束敏感度
2. **噪声添加**：向聚合梯度添加高斯噪声
3. **隐私预算**：跟踪累计隐私损失(ε, δ)

### 隐私-效用权衡
| ε值 | 隐私级别 | 模型精度影响 |
|------|---------|-------------|
| 1-3 | 强隐私 | 显著下降 |
| 3-8 | 中等隐私 | 中等下降 |
| 8+ | 弱隐私 | 轻微下降 |
| ∞ | 无隐私 | 无影响 |

**产业应用**：Apple/Google在设备端训练中使用本地差分隐私。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class DPSGD:
    def __init__(self, model, lr=0.01, noise_multiplier=1.0, max_grad_norm=1.0, delta=1e-5):
        self.model = model
        self.lr = lr
        self.noise_multiplier = noise_multiplier
        self.max_grad_norm = max_grad_norm
        self.delta = delta
        self.steps = 0
        self.cumulative_epsilon = 0

    def step(self, batch_size, dataset_size):
        total_norm = 0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm() ** 2
        total_norm = total_norm ** 0.5
        clip_coef = self.max_grad_norm / (total_norm + 1e-6)
        for p in self.model.parameters():
            if p.grad is not None:
                p.grad = p.grad * min(clip_coef, 1.0)
                noise = torch.randn_like(p.grad) * self.noise_multiplier * self.max_grad_norm
                p.data -= self.lr * (p.grad + noise)
        self.steps += 1
        self._update_privacy_budget(batch_size, dataset_size)

    def _update_privacy_budget(self, batch_size, dataset_size):
        q = batch_size / dataset_size
        sigma = self.noise_multiplier
        if sigma > 0:
            c2 = sigma ** (-2)
            term = q * math.sqrt(2 * self.steps * c2 * math.log(1 / self.delta))
            self.cumulative_epsilon = min(term, q * self.steps * c2)

    def get_privacy_spent(self):
        return self.cumulative_epsilon, self.delta

class StandardSGD:
    def __init__(self, model, lr=0.01):
        self.model = model
        self.lr = lr

    def step(self):
        for p in self.model.parameters():
            if p.grad is not None:
                p.data -= self.lr * p.grad

dp_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 5))
std_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 5))
std_model.load_state_dict(dp_model.state_dict())

dp_optimizer = DPSGD(dp_model, noise_multiplier=1.0, max_grad_norm=1.0)
std_optimizer = StandardSGD(std_model, lr=0.01)

x = torch.randn(32, 64)
y = torch.randint(0, 5, (32,))

print('=== DP-SGD vs Standard SGD ===')
dp_losses = []
std_losses = []
for epoch in range(20):
    dp_loss = F.cross_entropy(dp_model(x), y)
    dp_loss.backward()
    dp_optimizer.step(batch_size=32, dataset_size=1000)
    dp_model.zero_grad()
    dp_losses.append(dp_loss.item())

    std_loss = F.cross_entropy(std_model(x), y)
    std_loss.backward()
    std_optimizer.step()
    std_model.zero_grad()
    std_losses.append(std_loss.item())

    if (epoch + 1) % 5 == 0:
        eps, delta = dp_optimizer.get_privacy_spent()
        print(f'Epoch {epoch+1}: DP_loss={dp_loss.item():.4f}, Std_loss={std_loss.item():.4f}, ε={eps:.2f}')

eps, delta = dp_optimizer.get_privacy_spent()
print(f'\nFinal privacy budget: ε={eps:.2f}, δ={delta:.1e}')
print(f'Final loss: DP={dp_losses[-1]:.4f}, Standard={std_losses[-1]:.4f}')
print(f'\nKey: DP-SGD adds noise to protect privacy, but increases training loss.')
print(f'Higher noise_multiplier = stronger privacy = more accuracy loss.')
print(f'The ε value quantifies cumulative privacy loss over all training steps.')

## 2. 数据脱敏与匿名化

**数据脱敏**在训练前移除或替换敏感信息。

### 脱敏方法
- **PII检测与替换**：姓名→[NAME]，电话→[PHONE]，邮箱→[EMAIL]
- **K-匿名**：确保任意记录至少与K-1条其他记录在准标识符上相同
- **L-多样性**：在K-匿名基础上，确保每个等价类的敏感属性至少有L个不同值
- **T-接近**：敏感属性分布接近全局分布

### LLM特有挑战
- 模型可能"记住"训练数据中的PII
- 需要检测模型输出中的PII泄露
- 脱敏可能影响模型对特定实体的理解

In [ ]:
class PIIDetector(nn.Module):
    def __init__(self, d_model=128, n_pii_types=5):
        super().__init__()
        self.n_pii_types = n_pii_types
        self.pii_names = ['name', 'phone', 'email', 'address', 'id_number']
        self.detector = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_pii_types), nn.Sigmoid()
        )
        self.replacer = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, d_model))
            for _ in range(n_pii_types)
        ])
        self.pii_embeddings = nn.ParameterList([
            nn.Parameter(torch.randn(1, d_model) * 0.1)
            for _ in range(n_pii_types)
        ])

    def detect(self, text_embed):
        pii_scores = self.detector(text_embed)
        detected = []
        for i, score in enumerate(pii_scores[0]):
            if score.item() > 0.5:
                detected.append((self.pii_names[i], score.item()))
        return pii_scores, detected

    def redact(self, text_embed, pii_scores):
        redacted = text_embed.clone()
        for i, score in enumerate(pii_scores[0]):
            if score.item() > 0.5:
                replacement = self.pii_embeddings[i].expand_as(redacted)
                mask = score.item()
                redacted = redacted * (1 - mask) + replacement * mask
        return redacted

class KAnonymityChecker:
    def __init__(self, k=5, n_quasi_identifiers=3):
        self.k = k
        self.n_qi = n_quasi_identifiers

    def check(self, records):
        groups = {}
        for record in records:
            key = tuple(record[:self.n_qi].tolist())
            groups[key] = groups.get(key, 0) + 1
        violations = sum(1 for count in groups.values() if count < self.k)
        return len(groups), violations, violations / max(len(groups), 1)

pii_detector = PIIDetector(d_model=128)

normal_text = torch.randn(1, 128) - 0.3
pii_text = torch.randn(1, 128) + 0.5

print('=== PII Detection and Redaction ===')
for name, text in [('Normal', normal_text), ('PII-containing', pii_text)]:
    scores, detected = pii_detector.detect(text)
    print(f'{name}: detected={detected}')
    redacted = pii_detector.redact(text, scores)
    diff = (redacted - text).norm().item()
    print(f'  Redaction distance: {diff:.3f}')

checker = KAnonymityChecker(k=3, n_quasi_identifiers=2)
records = torch.randint(0, 5, (20, 4)).float()
n_groups, violations, violation_rate = checker.check(records)
print(f'\nK-Anonymity (k=3): {n_groups} groups, {violations} violations ({violation_rate:.1%})')

print(f'\nKey: PII detection + redaction removes sensitive info before training.')
print(f'K-anonymity ensures records cannot be uniquely identified.')

## 3. 遗忘学习 (Machine Unlearning)

**遗忘学习**使模型"忘记"特定训练数据，满足GDPR"被遗忘权"要求。

### 方法分类
| 方法 | 原理 | 速度 | 完整性 |
|------|------|------|--------|
| 重新训练 | 从头训练不含目标数据 | 慢 | 完美 |
| 梯度上升 | 对目标数据做梯度上升 | 快 | 不完美 |
| 影响函数 | 估计移除数据的影响 | 中 | 近似 |
| 知识编辑 | 修改模型中的特定知识 | 快 | 局部 |

### 挑战
- **验证困难**：如何证明模型确实"忘记"了？
- **级联效应**：移除一条数据可能影响其他知识的表示
- **隐私-效用权衡**：遗忘过多数据会损害模型性能

**产业实践**：GDPR要求、CCPA合规。

In [ ]:
class GradientAscentUnlearning:
    def __init__(self, model, lr=0.01, unlearning_strength=2.0):
        self.model = model
        self.lr = lr
        self.unlearning_strength = unlearning_strength

    def unlearn_step(self, x_forget, y_forget):
        logits = self.model(x_forget)
        loss = F.cross_entropy(logits, y_forget)
        for p in self.model.parameters():
            if p.grad is not None:
                p.data += self.lr * self.unlearning_strength * p.grad
        return loss.item()

class InfluenceUnlearning:
    def __init__(self, model, lr=0.01):
        self.model = model
        self.lr = lr
        self.stored_params = None

    def store_params(self):
        self.stored_params = {n: p.clone() for n, p in self.model.named_parameters()}

    def compute_influence(self, x_forget, y_forget, x_retain, y_retain):
        logits_forget = self.model(x_forget)
        loss_forget = F.cross_entropy(logits_forget, y_forget)
        grads_forget = torch.autograd.grad(loss_forget, self.model.parameters(), create_graph=True)
        logits_retain = self.model(x_retain)
        loss_retain = F.cross_entropy(logits_retain, y_retain)
        grads_retain = torch.autograd.grad(loss_retain, self.model.parameters(), create_graph=True)
        influence = sum((gf * gr).sum() for gf, gr in zip(grads_forget, grads_retain))
        return influence.item()

    def unlearn_step(self, x_forget, y_forget, x_retain, y_retain):
        logits_forget = self.model(x_forget)
        loss_forget = F.cross_entropy(logits_forget, y_forget)
        grads = torch.autograd.grad(loss_forget, self.model.parameters())
        with torch.no_grad():
            for p, g in zip(self.model.parameters(), grads):
                p.data += self.lr * g
        logits_retain = self.model(x_retain)
        loss_retain = F.cross_entropy(logits_retain, y_retain)
        loss_retain.backward()
        with torch.no_grad():
            for p in self.model.parameters():
                if p.grad is not None:
                    p.data -= self.lr * 0.5 * p.grad
                    p.grad = None
        return loss_forget.item(), loss_retain.item()

model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 5))
x_forget = torch.randn(8, 64)
y_forget = torch.randint(0, 5, (8,))
x_retain = torch.randn(32, 64)
y_retain = torch.randint(0, 5, (32,))

print('=== Machine Unlearning ===')
print(f'Before unlearning:')
with torch.no_grad():
    forget_acc = (model(x_forget).argmax(1) == y_forget).float().mean()
    retain_acc = (model(x_retain).argmax(1) == y_retain).float().mean()
print(f'  Forget set accuracy: {forget_acc:.2%}')
print(f'  Retain set accuracy: {retain_acc:.2%}')

unlearner = InfluenceUnlearning(model, lr=0.01)
unlearner.store_params()

print(f'\nUnlearning iterations:')
for step in range(10):
    f_loss, r_loss = unlearner.unlearn_step(x_forget, y_forget, x_retain, y_retain)
    if (step + 1) % 3 == 0:
        with torch.no_grad():
            forget_acc = (model(x_forget).argmax(1) == y_forget).float().mean()
            retain_acc = (model(x_retain).argmax(1) == y_retain).float().mean()
        print(f'  Step {step+1}: forget_acc={forget_acc:.2%}, retain_acc={retain_acc:.2%}')

print(f'\nKey: Unlearning reduces accuracy on forgotten data while preserving retained knowledge.')
print(f'Gradient ascent on forget set + gradient descent on retain set is a common approach.')
print(f'Verifying complete unlearning remains an open research challenge.')

## 4. 联邦学习与隐私计算

**联邦学习**在数据不出本地的前提下联合训练模型。

### 核心流程
1. 服务器分发全局模型给各客户端
2. 客户端在本地数据上训练
3. 客户端上传模型更新（梯度/权重差）
4. 服务器聚合更新（FedAvg等）
5. 重复1-4直到收敛

### 隐私增强技术
- **安全聚合**：服务器无法看到单个客户端的更新
- **本地差分隐私**：客户端在上传前添加噪声
- **同态加密**：在加密数据上直接计算
- **可信执行环境(TEE)**：在安全硬件中执行计算

### LLM联邦学习挑战
- 模型太大，通信成本高
- 异构数据导致客户端漂移
- 需要PEFT(如FedLoRA)降低通信量

In [ ]:
class FederatedClient:
    def __init__(self, client_id, d_model=64, n_classes=5, local_lr=0.01, local_steps=3, dp_noise=0.0):
        self.client_id = client_id
        self.model = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, n_classes))
        self.local_lr = local_lr
        self.local_steps = local_steps
        self.dp_noise = dp_noise

    def train_local(self, x, y):
        initial_params = {n: p.clone() for n, p in self.model.named_parameters()}
        for _ in range(self.local_steps):
            logits = self.model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            with torch.no_grad():
                for p in self.model.parameters():
                    if p.grad is not None:
                        if self.dp_noise > 0:
                            p.grad += torch.randn_like(p.grad) * self.dp_noise
                        p.data -= self.local_lr * p.grad
                        p.grad = None
        update = {}
        for n, p in self.model.named_parameters():
            update[n] = p.data - initial_params[n]
        return update

class FederatedServer:
    def __init__(self, d_model=64, n_classes=5, n_clients=3):
        self.d_model = d_model
        self.n_classes = n_classes
        self.global_model = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, n_classes))
        self.n_clients = n_clients
        self.round = 0

    def distribute(self, clients):
        for client in clients:
            client.model.load_state_dict(self.global_model.state_dict())

    def aggregate(self, updates):
        avg_update = {}
        for name in updates[0]:
            avg_update[name] = sum(u[name] for u in updates) / len(updates)
        with torch.no_grad():
            for n, p in self.global_model.named_parameters():
                p.data += avg_update[n]
        self.round += 1

n_clients = 3
clients = [FederatedClient(i, dp_noise=0.1) for i in range(n_clients)]
server = FederatedServer(n_clients=n_clients)

client_data = [(torch.randn(16, 64) + i * 0.3, torch.randint(0, 5, (16,))) for i in range(n_clients)]

print('=== Federated Learning ===')
for round_num in range(5):
    server.distribute(clients)
    updates = []
    for i, client in enumerate(clients):
        x, y = client_data[i]
        update = client.train_local(x, y)
        updates.append(update)
    server.aggregate(updates)
    test_x = torch.randn(32, 64)
    test_y = torch.randint(0, 5, (32,))
    with torch.no_grad():
        acc = (server.global_model(test_x).argmax(1) == test_y).float().mean()
    print(f'Round {round_num+1}: global_acc={acc:.2%}')

print(f'\nKey: Federated learning trains on distributed data without centralizing it.')
print(f'Local DP noise protects individual client contributions.')
print(f'Communication cost is the main bottleneck for LLM federated learning.')

## 5. LLM隐私风险评估

### 主要风险
1. **训练数据提取**：通过特定提示诱导模型输出训练数据
2. **成员推断**：判断特定数据是否在训练集中
3. **属性推断**：推断训练数据的统计属性
4. **PII泄露**：模型输出中包含个人身份信息

### 防御措施
| 风险 | 防御方法 |
|------|----------|
| 训练数据提取 | DP训练、输出过滤、温度降低 |
| 成员推断 | DP训练、正则化、数据增强 |
| 属性推断 | DP训练、数据最小化 |
| PII泄露 | 数据脱敏、输出审查、PII检测 |

### 合规要求
- **GDPR**：欧盟，被遗忘权、数据最小化
- **CCPA**：加州，消费者数据权利
- **AI Act**：欧盟，高风险AI系统透明度要求

In [ ]:
class PrivacyRiskAssessor:
    def __init__(self, d_model=128):
        self.d_model = d_model
        self.extraction_detector = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.membership_attack = nn.Sequential(
            nn.Linear(d_model * 2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.pii_scanner = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 4), nn.Sigmoid()
        )
        self.pii_types = ['name', 'email', 'phone', 'address']

    def assess_extraction_risk(self, output_embed):
        return self.extraction_detector(output_embed)

    def assess_membership_risk(self, data_embed, model_output_embed):
        features = torch.cat([data_embed, model_output_embed], dim=-1)
        return self.membership_attack(features)

    def scan_pii(self, output_embed):
        scores = self.pii_scanner(output_embed)
        detected = []
        for i, score in enumerate(scores[0]):
            if score.item() > 0.5:
                detected.append((self.pii_types[i], score.item()))
        return scores, detected

    def full_assessment(self, output_embed, data_embed=None):
        extraction_risk = self.assess_extraction_risk(output_embed)
        pii_scores, pii_detected = self.scan_pii(output_embed)
        membership_risk = None
        if data_embed is not None:
            membership_risk = self.assess_membership_risk(data_embed, output_embed)
        overall_risk = extraction_risk.item()
        if pii_detected:
            overall_risk = max(overall_risk, max(s for _, s in pii_detected))
        if membership_risk is not None:
            overall_risk = max(overall_risk, membership_risk.item())
        return {
            'extraction_risk': extraction_risk.item(),
            'pii_detected': pii_detected,
            'membership_risk': membership_risk.item() if membership_risk is not None else None,
            'overall_risk': overall_risk,
            'risk_level': 'HIGH' if overall_risk > 0.7 else 'MEDIUM' if overall_risk > 0.4 else 'LOW'
        }

assessor = PrivacyRiskAssessor(d_model=128)

safe_output = torch.randn(1, 128) - 0.3
risky_output = torch.randn(1, 128) + 0.5
data = torch.randn(1, 128)

print('=== Privacy Risk Assessment ===')
for name, output in [('Safe output', safe_output), ('Risky output', risky_output)]:
    result = assessor.full_assessment(output, data)
    print(f'\n{name}:')
    print(f'  Extraction risk: {result["extraction_risk"]:.3f}')
    print(f'  PII detected: {result["pii_detected"]}')
    print(f'  Membership risk: {result["membership_risk"]:.3f}')
    print(f'  Overall risk: {result["overall_risk"]:.3f} ({result["risk_level"]})')

print(f'\nKey: Privacy risk assessment should be integrated into the model deployment pipeline.')
print(f'Multiple risk dimensions need to be evaluated simultaneously.')
print(f'Compliance requirements (GDPR, CCPA) mandate privacy-by-design.')

## 6. 隐私保护总结

| 技术 | 保护目标 | 数学保证 | 性能影响 |
|------|---------|---------|----------|
| DP-SGD | 训练数据隐私 | (ε,δ)-DP | 精度下降 |
| 数据脱敏 | PII保护 | 无形式化保证 | 信息损失 |
| K-匿名 | 身份保护 | K-不可区分 | 数据粒度损失 |
| 遗忘学习 | 被遗忘权 | 不完美 | 级联影响 |
| 联邦学习 | 数据不离开本地 | 无（需结合DP） | 通信成本高 |
| 安全聚合 | 客户端更新隐私 | 信息论安全 | 通信开销 |

**最佳实践**：
1. **隐私设计**：从系统设计阶段就考虑隐私
2. **数据最小化**：只收集必要的数据
3. **多层防护**：DP + 脱敏 + 输出过滤
4. **持续审计**：定期评估隐私风险
5. **合规优先**：确保满足GDPR/CCPA要求